# Baseball Lab 9: Correlation and regression models

Welcome to the ninth lab!  In today's lab you'll get practice:

1. Examining the concept of *regression to the mean* 
2. Seeing how to fit polynomial functions to data 
3. Exploring Bill James "Pythagorean Expectation" and discussing cross-validation


#### Deadline

This assignment is due **Sunday April 12th at 11pm**. You can turn in the assignment up to 24 hours late for 90% credit (after that, the homework will only be accepted with a Dean's Extension). Directly sharing answers is not okay, but discussing problems with the course staff or with other students is encouraged. Refer to the policies page to learn more about how to learn cooperatively. You should start early so that you have time to get help if you're stuck. If you have questions, please post them to [Ed Discussion](https://edstem.org/us/courses/89102/discussion) (and answer others' questions too). 




## Getting started - downloading the data

In order to complete this lab, it is necessary to download a few files. Please run the code below **only once** to download data needed to complete the lab. To run the code, click in the cell below and press the play button (or press shift-enter). 


In [11]:
# Please run this code once to download the files you will need to complete the homework 

import YData_baseball

YData_baseball.download_image("SI_cover_jinx.jpg")
YData_baseball.download_data("Lahman_2025/Batting.csv") 
YData_baseball.download_data("Lahman_2025/AwardsPlayers.csv")
YData_baseball.download_data("Lahman_2025/Teams.csv")


The file `SI_cover_jinx.jpg` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.
The file `Batting.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.
The file `AwardsPlayers.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.
The file `Teams.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.


# Part 0: Quote and reaction to Astroball chapter 9 (5 points)

Please find an interesting quote from chapter 9 of Astroball and then write a ~one paragraph reaction to the quote below.

*Quote:*  ...

Reaction: ... 

# Useful packages and functions

The code below loads the packages we will need, and defines the `add_derived_stats()` and `get_rmse()` functions from previous labs. You will use these functions later in this lab to add additional statistics to your data and to calculate the RMSE. 

In [12]:
# importing packages that will be used in this lab

import pandas as pd
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf 


np.random.seed(1730)


# A function to calculate the RMSE (from lab 8)
def get_rmse(the_data, y_col, x_col):
    
    the_formula = y_col + '~' + x_col
    
    lm = smf.ols(the_formula, data = the_data).fit()

    predictions = lm.predict(the_data).to_numpy()
   
    return (np.mean((the_data[y_col] - predictions)**2))**.5




# A function that adds that statistics: BA, SLG, OBP, and OPS
def add_derived_stats(teams_data):
    
    # batting average: ba
    ba = teams_data['H']/teams_data['AB']

    # create slugging percentage
    x1b = teams_data['H'] - (teams_data['2B'] + teams_data['3B'] + teams_data['HR'])
    slg = (x1b + (2 * teams_data['2B'] + 3 * teams_data['3B'] + 4 * teams_data['HR']))/teams_data['AB']

    # create on-base percentage
    obp = (teams_data['H'] + teams_data['BB'] + teams_data['HBP'])/(teams_data['AB'] + teams_data['BB'] + teams_data['HBP'] + teams_data['SF'])

    # create on-base plus slugging percentage
    ops = obp + slg

    # create the table that has all the desired statistics
    teams_with_addition_stats = teams_data.copy()
    teams_with_addition_stats["1B"] = x1b
    teams_with_addition_stats["BA"] = ba
    teams_with_addition_stats["SLG"] = slg
    teams_with_addition_stats["OBP"] = obp
    teams_with_addition_stats["OPS"] = ops
    
    return teams_with_addition_stats


# Part 1: Regression to the mean - explaining the Rookie of the Year Curse


As discussed in the class and on [wikipedia](https://en.wikipedia.org/wiki/Regression_toward_the_mean), "Sir Francis Galton observed that extreme characteristics (e.g., height) in parents are not passed on completely to their offspring. Rather, the characteristics in the offspring regress towards a mediocre point (a point which has since been identified as the mean)." In fact, Galton first coined the term "regression", for predicting a y value based on x values, based on the fact that the heights of children of tall (and short) parents tend to regress toward the average height of the population. 

A similar effect can be seen in baseball. For example, there is the well-known [Sports Illustrated Cover Jinx](https://en.wikipedia.org/wiki/Sports_Illustrated_cover_jinx), where players tend to perform worse after they appear on the cover of Sports Illustrated Magazine. There is also the "Rookie of the Year Curse" where players tend to perform worse after receiving the Rookie of the Year award. In the following exercises, we will examine this phenomenon to understand what is driving it. 

<img src="SI_cover_jinx.jpg">



## 1.1: Examining regression to the mean by simulating two seasons 

To understand the regression to the mean phenomenon, we will simulate the **performance** (statistics) of a set of baseball players based on their underlying **abilities** (parameters). 

The function written below called `simulate_players_stats(batting_abilities, n)` simulates players' performance statistics, based on their underlying abilities. The arguments to this function are: 

1. `batting_abilities`: an ndarray of proportions ($\pi$'s) specifying the abilities of a set of players. For example, it could be a 50 element ndarray of proportions specifying the OBP abilities of 50 players. Note: we would never know these real abilities perfectly in the "real world" but for the sake of simulation we can specify them perfectly. 
    
2. `n`: this lists the number of events to simulate for each player. For example, if we simulate OBP, then n would specify how many plate appearances we want to simulate for each player.

The `simulate_players_stats()` function returns simulated performance statistics ($\hat{p}$'s) from these players for n events. 

Note: I am sure you could easily write the `simulate_players_stats()` function yourself, but to give you more time to work on your final project I have written it for you :)


**Exercise 1.1 (10 points)**:  Let's use the `simulate_players_stats()` to simulate players' observed batting averages (BA), based on their true batting average abilities. We will do this simulation for two seasons and see if we observe the regression to the mean phenomenon, e.g., players who have high BA's in the first season tend to have lower BAs in the second season, etc. 

To do this simulation, please complete the following steps: 

1. Use the `np.arange()` to create an ndarray called `batting_abilities` that has players' true BA abilities. The abilities should range from .250 to .320 in steps of .001, i.e, this ndarray should list the abilities of a total of 70 players. 

2. Use the `simulate_players_stats()` function to create the observed performances for one season of 70 players based on the abilities of these 70 players as specified in the `batting_abilities` ndarray. Use `n = 500` plate appearances for these players and save the results to the name `season_1`.

3. Repeat step 2 to simulate the BA performance for a second season and save it to the name `season_2`

4. Next create a scatter plot of the two seasons from this DataFrame

5. Add a red regression line to the scatter plot. To do this, use `b1, b0 = np.polyfit(season_1, season_2, 1)` to get the slope and intercept for the regression line, and then use `plt.plot(x, b1*x + b0)` to add the line to the plot. Note: you will need to create an ndarray called `x` that has values that range from the minimum to maximum of the data in `season_1` to get the line to show up on the plot. 

6. Add the identity line (line of slope 1, intercept of 0) that shows how well players would do if they performed equally well on both seasons. To do this you can create a ndarray of abilities ranging from .200 to .380 at increments of .001 and then use the `plt.plot(x, x)` function to create this line. Make your identity line green and your regression line red to make it easy to see the difference between these two lines. 

In the answer section, report if this plot shows the regression to the mean phenomenon and explain why this is happening. 


In [13]:

def simulate_players_stats(batting_abilities, n):
    
    all_stats = []
    
    for i in np.arange(0, len(batting_abilities)):
        
        prob_heads = batting_abilities[i]
        
        curr_stat = np.random.binomial(n, prob_heads)/n 
        
        all_stats.append(curr_stat)
        
    return all_stats



# create an array of batting abilities for 70 players ranging from .250 to .320 in steps of .001



# simulate the observed batting averages for two seasons based on the true abilities of the players



# create a scatter plot of the two seasons and add a regression line and an identity line to this plot



# add the regression line to the plot



# add the identity line to the plot



# label your axes!




<b><p style="color:red">Answer</p></b>







## 1.2 Does the Rookie of the Year Curse exist? 

Above we showed that regression to the mean exists in a simple simulation, but does it actually occur in the real baseball data? While many people claim it does, can we actually see it in the data? Since it is going to be hard to find all the covers of Sports Illustrated that players were on, let's focus on the "Rookie of the Year Curse" to see if we can see it there.

**Exercise 1.2 (12 points)**: The code below loads a DataFrame of the list of awards players have won in their careers and also the season level batting data from all players. Please create a scatter plot that, for each player who won rookie of the year, shows their OPS in their rookie season on the x-axis and their OPS in their second year on the y-axis.  Only include players who had an OPS greater than 0.6 in both their rookie and second year seasons to get rid of missing data and pitchers, etc. Also, add the identity line to this plot as well to get a sense of players who performed worse their second season. 

In the answer section below, report the percentage of players who won rookie of the year award and had a worse OPS statistic in their second year (again, only use players who have OPS that are greater than 0.6 for both their rookie and second year seasons). Does this plot suggest the regression to the mean phenomenon is actually occuring? 


In [14]:
# read in a table of the awards and batting statistics
awards = pd.read_csv("AwardsPlayers.csv")
batting = pd.read_csv('Batting.csv')  
batting = add_derived_stats(batting)  # add derived statistics including OPS





# get rookie of the year statistics



# get statistics for the second year players played




# join the rookie_stats DataFrame and the second_year_stats DataFrame




# join the first and second year DataFrames, and relabel the columns




# create the scatter plot






# print the proportion of players who performed worse in their second year




<b><p style="color:red">Answer</p></b>

70% of players who won rookie of the year performed worse in their second season.  Players in general should be getting better in their second season, so this indeed suggests that the of regression to the mean phenomenon is occurring - although it might be good to run a hypothesis test to be sure that the value of 70% is above the 50% we would expect by chance.

# Part 2: Non-linear functions of the predictors

To improve the fit of our linear models to the data, we can also include additional predictors that are derived from the original predictions in our data set. For example, we might want to predict the amount of runs a team will score as a function of the number of home runs and home runs squared, which would result in the equation: $\hat{r} =  \hat{\beta}_0 + \hat{\beta}_1 \cdot HR + \hat{\beta}_2 \cdot HR^2$. This function still creates a linear combination of predictors, but since some of the predictors are now non-linear functions of the original predictors, the final prediction function is non-linear in the original predictors. 



## 2.1: Quadratic fits for predicting maximum BA as a function of the year

Let's examine a quadratic fit for predicting the maximum batting average in a season as a function of the year. 

**Exercise 2.1 (12 points)**: Please complete the following steps:

1. Create a DataFrame called `ba_max_year` that has maximum batting average for each year from 1920 to the present for all players who had more than 502 at-bats. Hint: the `df.groupby()` method will be useful here. 

2. Add a column to the `ba_max_year` DataFrame called `year2` that has the values of the years squared. 

3. Using the statsmodel package, fit a linear model predicting the batting average as a function of the year and save it to the name `fit_lin`. Also print out the coefficients of this linear model. 

4. Create a scatter plot of the maximum BA as a function of the year. Add to this scatter plot, a line for the predictions made by the linear model. Hint: the `fit_lin.predict()` method will be useful here.

5. Repeat steps 3 and 4 but also add the quadratic year term and save the model to the name `fit_quad`. Again, print the coefficients and add a line for the quadratic fit. Make sure this is on a new figure by using the `plt.figure()` prior to creating the scatter plot and adding the quadratic fit line.

6. Use the `get_rmse()` function to get the RMSE for both models. In the answer section below, describe which model you think has the better fit. 



In [15]:
# get the batting data an add the derived statistics
batting = pd.read_csv('Batting.csv')  
batting = add_derived_stats(batting)


# create the ba_max_year DataFrame




# add the year squared column to this DataFrame



# fit a linear model and print the coefficients





# create a scatter plot of the data and add the linear fit to this plot





# fit a quadratic model and print the coefficients and plot it




# create a new figure and plot the quadratic fit on top of a scatter plot of the data






# print the RMSE for both models




<b><p style="color:red">Answer</p></b>



## 2.2: Adding higher order polynomial terms

Let's now extend the model to include higher order terms up to degree 6.

**Exercise 2.2 (8 points)**: Please create a degree 6 model of the form: $\hat{BA} =  b_0 + b_1 \cdot year + b_2 \cdot year^2 + b_3 \cdot year^3 ... + b_6 \cdot year^6$. Again, print the coefficients of this model, create a scatter plot that contains the fitted line, and print the RMSE. In the answer section below, state whether you think this model is a better fit than the quadratic model. 


In [16]:
# add the year3, year4, year5, and year6 columns to the ba_max_year DataFrame





# fit the 6th degree model, print the coefficients 




# visualize the 6th degree model fit





# print the RMSE for the model





<b><p style="color:red">Answer</p></b>




# Part 3: Bill James "Pythagorean expectation" and cross-validation


As we have previously discussed, one of the founders of [sabermetrics](https://en.wikipedia.org/wiki/Sabermetrics) was [Bill James](https://en.wikipedia.org/wiki/Bill_James). One of the many contributions Bill James made to the analysis of baseball data was his ["Pythagorean expectation"](https://en.wikipedia.org/wiki/Pythagorean_expectation) which predicts a team's win to loss ratio (W/L) at the end of a season as a function of the number of runs scored (R) and the number of rows allowed (RA) in a season. More specifically, the formula is: $$\frac{W}{L} = (\frac{R}{RA})^2$$
 
In these exercises we will examine the Pythagorean expectation formula and see if we can come up with a slightly better formula for predicting the win to loss ratio. We will also use cross-validation to make sure that the formula we come up with works better for predicting new data rather than just overfitting to the data we have. 


## 3.1: Assessing how well the Pythagorean expectation works

Let's start by examining how well the Pythagorean expectation formula works for predicting the win to loss ratio. 


**Exercise 3.1 (12 points)**:  The code below loads data for teams since 1970 that have played 162 games. Please evaluate how well the Pythagorean expectation formula works by completing the following steps: 

1. Add a column to the `teams_162` DataFrame called `'WL_ratio'` that has the win loss ratio for all teams.

2. Add a column to the `teams_162` DataFrame called `'Pythag_prediction'` that has the Pythagorean expectation predicted value for all teams; i.e., that has the value $(R/RA)^2$.

3. Create a scatter plot of win loss ratio as a function of the Pythagorean expectation value. 

4. Fit a least squares model to predict the loss ratio as a function of the Pythagorean expectation and print out the slope and intercept found.

5. Add the regression line to your scatter plot based on the slope and intercept you found in step 4.

6. In the answer section below, report what the slope and intercept should be if the Pythagorean expectation was a perfect fit for the data.

7. Finally print the RMSE for that the Pythagorean expectation formula gives using the `get_rmse()` function.


In [17]:
teams = pd.read_csv('Teams.csv')  
teams_162 = teams[(teams.yearID > 1969) & (teams.G == 162)].copy()


# add columns the teams_162 data for the win loss ratio and the Pythagorean expectation predictions




# create a scatter plot of the loss ratio and the Pythagorean expectation predictions




# fit a linear model to predict the loss ratio as a function of the Pythagorean expectation and print out the slope and intercept found




# add the regresison line to your plot




# print the RMSE to see how good this formula is




<b><p style="color:red">Answer</p></b>



## 3.2:  Improving the Pythagorean expectation formula

Let's now see if we can improve on the Pythagorean expectation formula. In particular, the exponent on the runs to runs allowed ratio (R/RA) is 2; i.e., the formula is $\frac{W}{L} = (\frac{R}{RA})^2$. But perhaps there is a better exponent that should be used rather than 2; i.e., perhaps there is a value $k$ that would give a better fit in the formula: $\frac{W}{L} = (\frac{R}{RA})^k$. 

To find the value $k$ that works best, we can take the log of the general Pythagorean expectation formula: $log(\frac{W}{L}) = log((\frac{R}{RA})^k) = k \cdot log(\frac{R}{RA})$. We can then fit a linear model for $y = log(\frac{W}{L})$ as a function of $x = log(\frac{R}{RA})$ to find the optimum value for $k$. 

We should note that in fitting the above model there is no intercept term. We can fit a model without an intercept term using the statsmodels formula interface by including a -1 in our formula. i.e., using the syntax: `smf.ols('y ~ x -1', data = my_data)`.

**Exercise 3.2 (12 points)**: Let's try to find the optimal exponent *k* by completing the following steps: 

1. Create a new DataFrame called `teams_162_transform` that starts with the `teams_162` table and adds a column called `log_WL_ratio` which has the log of the wins to losses

2. Add a column `log_R_RA_ratio` to the `teams_162_transform` DataFrame which is the log of R/RA. 

3. Fit a linear regression model for predicting the `log_WL_ratio` as a function of `log_R_RA_ratio`. Make sure the model only has a slope and no intercept. Print the value of the slope for this model. 

4. Add a column called `better_Pythag_prediction` to the `teams_162_transform` DataFrame that contains the predictions for the new linear regression model using the "optimal" *k* found in step 3. 

5. Get the RMSE from using the model with the optimum *k*. Is the RMSE smaller with the optimal k? 

In the answer section, report what the optimal k is, what the RMSE is with this optimal k, and what the RMSE is with the original exponent of 2.



In [18]:
# add the 'log_WL_ratio' and 'log_R_RA_ratio' columns to the teams_162_transform table



# fit a linear model between log_WL_ratio and log_R_RA_ratio without the intercept


# print the regression slope


# add 'better_Pythag_prediction' column to the teams_162_transform table


# get the new RMSE based on using the "optimal" exponent k



<b><p style="color:red">Answer</p></b>



## 3.3: Using cross-validation to assess if the model with the optimal k really makes better predictions

In exercise 3.2 we found the "optimal k" and assessed how well the model fit (via the RMSE) using the same data. Using the same data to fit and assess a model is generally a bad idea because this can lead to "overfitting" where the model is really just fitting the noise in the data rather than a true underlying relationship. 

**Exercise 3.3 (10 points)**:   To assess whether the new model really is better at making predictions, we can use cross-validation where we fit the model on one subset of data and then evaluate how well the model makes predictions (via the RMSE) on a new data set. Let's try this now by fitting the model on data from 2000 to 2009 and making predictions on data from 2010 to 2018 using the following steps: 

1. Create a DataFrame called `data_2000s` from the `teams_162_transform` DataFrame that contains only data from 2000 to 2009.

2. Create a DataFrame called `data_2010s` from the `teams_162_transform` DataFrame that contains only data from 2010 to 2019.

3. Print the RMSE for the data from 2010's using the Pythagorean expectation formula (i.e., with the exponent of 2). 

4. Fit a model of `log_WL_ratio` as a function of `log_R_RA_ratio` using data from only 2000's. Print the value for the exponent for this model.

5. Add a column called `better_Pythag_prediction2` to the `data_2010s` DataFrame that has the predictions based on using the coefficient *k* found from using the data from 2000's (in step 4). 

6. Get the RMSE for the `log_WL_ratio` in 2010's based on the `better_Pythag_prediction2` column. In the answer section below, report whether the RMSE for the model fit with the optimal $k$ found using data from the 2000's makes better predictions on the data from the 2010's compared to using the default Pythagorean expectation formula that has $k$ = 2. 

In [19]:
# get two DataFrames with data from 2000-2009 and with data from 2010-2018



# get the RMSE for the data from 2010 using the Pythagorian formula



# fit a model of log_WL_ratio as a function of log_R_RA_ratio using data from 2000's




# add a column called 'better_Pythag_prediction2' with the predictions using the coefficient found from data from 2000's





<b><p style="color:red">Answer</p></b>



# Part 4: Final project plan (3 points)

Please write a paragraph or two describing how progress on your project is going. 




# 5. Reflection (3 points)

Please fill out the lab 9 reflection on Canvas to let us know how this lab, and the class overall, is going for you. 



# 6.  Submitting your work

Once you're finished filling in and running all cells, you should submit your assignment as a pdf on Gradescope. You can access Gradescope through Canvas on the left-side of the class home page. The problems in each lab assignment are numbered. When submitting on Gradescope, please **make sure to select the correct pages of your pdf that correspond to each problem**. Failure to mark pages correctly **will result in points being deducted** from your score.

To convert this Jupyter notebook document to a pdf please run the code in the cell below. This should produce a pdf document which should appear in the files tab on the left (you might need to refresh the files tab to see this file). You can then right click on this file (command click on a mac), to download this pdf document which you can upload to Gradescope. 

Please be sure to check that all the code and output are visible before submitting your pdf to Gradescope as **points will be deducted for missing code and output that is not visible** since we will not be able to grade this.

In [20]:
%%capture

!quarto render lab_09.ipynb --cache-refresh --to pdf 


#### Alternative submission instructions

If converting your Jupyter notebook to a pdf using the command in the cell above does not work, an alternative way to convert your Jupyter notebook is:

1.  Go to "File" at the top-left of your Jupyter Notebook
2.  Under "Download as" (or "Save and Export Notebook As...") and select "HTML (.html)"
3.  After the .html has downloaded, open it and then select "File" and "Print" (note you will not actually be printing)
4.  From the print window, select the option to save as a .pdf
